# Personnes traitées pour hypertension artérielle : profil, recours aux soins et coûts

**Master en Sciences Biomédicales, Université de Genève**
Module d'économie de la santé. Travaux pratiques : analyse descriptive de données de remboursement de l'assurance obligatoire des soins.

## Question de recherche

Quel est le profil des personnes traitées pour hypertension artérielle parmi les assuré·es d'un assureur-maladie, comment recourent-elles au système de soins, et quels coûts représentent-elles pour l'assurance obligatoire des soins en comparaison des autres assuré·es ?

Avant d'y répondre, on décrit la population assurée et la répartition des dépenses : qui coûte, et pourquoi.

## Objectifs

À l'issue de ces travaux pratiques, les étudiant·es seront capables de décrire les principaux déterminants des dépenses de santé (âge, sexe, cause des prestations) et leur concentration, d'interpréter une analyse descriptive simple de données de remboursement, d'en identifier les principales limites, et d'en tirer des éléments d'interprétation à l'intention d'un·e décideur·e de santé publique.

## Données

Les données sont une extraction des décomptes 2024 d'un assureur-maladie de Suisse romande, pour 15 000 adultes. Chaque prestation indique sa cause (maladie, accident ou maternité) et le prestataire qui la facture. Les données sont **entièrement synthétiques**. Elles ont été simulées pour ressembler à des données réelles et ne concernent aucune personne existante. Les variables sont décrites dans le fichier `dictionnaire_donnees.md`. Le fichier des médicaments ne contient qu'une sélection de substances.

## Travail attendu

Chaque groupe rédige une note d'une page destinée à un·e responsable de santé publique, présentant la population concernée, les traitements et le recours aux soins, les coûts associés, deux ou trois messages principaux et une limite d'interprétation.

## Organisation du document

Les cellules s'exécutent dans l'ordre (Maj + Entrée). Le code est fourni, et il n'est pas nécessaire de le comprendre en détail : les commentaires (lignes qui commencent par `#`) disent à quoi sert chaque cellule. Chaque section se termine par des questions à discuter en groupe, qui serviront pour la note. Les paramètres modifiables sont signalés, et les analyses complémentaires sont facultatives.

Les **exercices** demandent de compléter quelques lignes de code par analogie avec les cellules qui précèdent : sélectionner les lignes de la cellule d'exercice, les décommenter (Ctrl + / ou Cmd + /), puis remplacer chaque `___` par le code attendu. Un indice est disponible sous chaque exercice.

Les **pièges d'interprétation** (A à D) présentent un graphique qui semble appeler une conclusion évidente. Il faut expliquer pourquoi les données de remboursement ne permettent pas de la tirer, avant d'exécuter la vérification.

Les réponses aux questions des exercices et des pièges sont notées avec celles de la section, dans la cellule « Réponses du groupe ». Les exercices facultatifs sont destinés aux groupes qui avancent plus vite.

| Section | Contenu | Exercices | Pièges | Durée indicative |
|---|---|---|---|---|
| 1 | Prise en main et qualité des données | 1 | | 20 min |
| 2 | Qui sont les assuré·es ? | 2 (3 facultatif) | | 20 min |
| 3 | Qui coûte, et pourquoi ? | 4 (5 et 6 facultatifs) | A, B | 45 min |
| 4 | Personnes traitées pour hypertension : identification, profil, traitements, recours aux soins, coûts | 7, 9, 10 (8 facultatif) | C, D | 75 min |
| 5 | Synthèse et rédaction de la note | | | 20 min |

<details><summary><b>Vous connaissez R plutôt que Python ?</b> Équivalences entre dplyr et pandas</summary>

| Opération | R (dplyr) | Python (pandas) |
|---|---|---|
| Premières lignes | `head(assures)` | `assures.head()` |
| Filtrer des lignes | `filter(assures, sexe == "F")` | `assures[assures["sexe"] == "F"]` |
| Deux conditions | `filter(assures, sexe == "F", age >= 65)` | `assures[(assures["sexe"] == "F") & (assures["age"] >= 65)]` |
| Choisir des colonnes | `select(assures, sexe, canton)` | `assures[["sexe", "canton"]]` |
| Créer une variable | `assures <- mutate(assures, age = 2024 - annee_naissance)` | `assures["age"] = 2024 - assures["annee_naissance"]` |
| Compter | `count(assures, canton)` | `assures["canton"].value_counts()` |
| Résumer par groupe | `prestations %>% group_by(cause) %>% summarise(total = sum(cout_chf))` | `prestations.groupby("cause")["cout_chf"].sum()` |
| Joindre deux tables | `left_join(prestations, assures, by = "id_assure")` | `prestations.merge(assures, on="id_assure", how="left")` |
| Trier | `arrange(prestations, desc(cout_chf))` | `prestations.sort_values("cout_chf", ascending=False)` |
| Valeurs manquantes | `is.na(x)` | `x.isna()` |

Quelques autres différences :
- les positions commencent à 0 (`assures.iloc[0]` est la première ligne),
- `True` et `False` s'écrivent avec une seule majuscule,
- quand on combine des conditions avec `&`, chacune va entre parenthèses.

</details>

In [ ]:
# @title Préparation de l'environnement (à exécuter sans modification)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}".replace(",", "'") if abs(x) >= 100 else f"{x:.1f}")

ANNEE = 2024
GROUPES = ["HTA traitée", "Autres assuré·es"]
COULEURS = {"HTA traitée": "#C0392B", "Autres assuré·es": "#7F8C8D"}          # un groupe mis en évidence, le reste en gris
COULEURS_SEXE = {"F": "#EB6834", "M": "#2A78D6"}
COULEURS_CAUSE = {"maladie": "#2A78D6", "maternite": "#EB6834", "accident": "#1BAF7A"}
CATEGORIELLES = ["#2A78D6", "#EB6834", "#1BAF7A", "#EDA100", "#E87BA4", "#008300", "#6250D6", "#E34948"]
RAMPE = ["#86B6EF", "#3987E5", "#1C5CAB", "#0D366B"]                          # catégories ordonnées, du clair au foncé
CLASSES_OFSP = [19, 26, 36, 46, 56, 66, 76, 86, 120]                         # classes d'âge de la statistique fédérale
LIBELLES_OFSP = ["19-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76-85", "86+"]

LIBELLES = {   # libellés des codes affichés dans les graphiques
    "standard": "Standard", "medecin_de_famille": "Médecin de famille", "hmo": "HMO", "telmed": "Télémédecine",
    "medecin_generaliste": "Généraliste", "medecin_specialiste": "Spécialiste", "ambulatoire_hospitalier": "Ambulatoire hospitalier",
    "physiotherapie": "Physiothérapie", "gastro_enterologie": "Gastro-entérologie", "gynecologie": "Gynécologie",
    "orthopedie": "Orthopédie", "orl": "ORL", "franchise_chf": "Franchise (CHF)", "modele_assurance": "Modèle d'assurance",
}


def libelle(code):
    """Libellé lisible d'un code."""
    return LIBELLES.get(code, str(code).replace("_", " ").capitalize())


def chf(x):
    """Montant en CHF avec séparateur de milliers."""
    return f"{x:,.0f}".replace(",", "'")


def axe_pct(ax, axe="y"):
    """Affiche un axe en pourcentage."""
    (ax.yaxis if axe == "y" else ax.xaxis).set_major_formatter(mtick.PercentFormatter(1, decimals=0))


def axe_chf(ax, axe="y"):
    """Affiche un axe en CHF avec séparateur de milliers."""
    formateur = mtick.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", "'"))
    (ax.yaxis if axe == "y" else ax.xaxis).set_major_formatter(formateur)


print("Environnement prêt.")

In [ ]:
# @title Chargement des données
# Enseignant·e : adresse du dossier contenant les fichiers CSV (ex. dépôt GitHub, lien « raw »)
URL_DONNEES = "https://raw.githubusercontent.com/daderidd/heco-tp-hypertension/main/donnees/"


def lire(nom, **options):
    local = Path("donnees") / nom                       # dossier « donnees » déposé dans Colab
    source = local if local.exists() else URL_DONNEES + nom
    return pd.read_csv(source, **options)


assures = lire("assures.csv", parse_dates=["debut_couverture", "fin_couverture"])
prestations = lire("prestations.csv", parse_dates=["date_prestation", "date_decompte"])
medicaments = lire("medicaments.csv", parse_dates=["date_delivrance"])
referentiel = lire("referentiel_atc.csv")
prestataires = lire("referentiel_prestataires.csv")

print(f"assures      : {len(assures):>8,} lignes")
print(f"prestations  : {len(prestations):>8,} lignes")
print(f"medicaments  : {len(medicaments):>8,} lignes")

---
## 1. Prise en main et qualité des données

Les données comprennent trois tables principales, reliées par l'identifiant `id_assure`, et deux référentiels :

```
assures                     1 ligne par personne assurée
   ├── prestations          1 ligne par position de facture (cause, type de prestation, prestataire)
   └── medicaments          1 ligne par médicament délivré (code ATC, prestataire)
referentiel_atc             code ATC → substance et groupe thérapeutique
referentiel_prestataires    id_prestataire → catégorie, canton, spécialité, statut juridique
```

In [ ]:
# Les 5 premières lignes de chaque table
display(assures.head())
display(prestations.head())
display(medicaments.head())

Les données administratives contiennent souvent des erreurs et des incohérences liées à la facturation. On commence donc par quelques contrôles.

In [ ]:
assures["age"] = ANNEE - assures["annee_naissance"]          # âge au 31 décembre
colonnes_prest = [c for c in prestations.columns if c != "id_ligne"]   # tout sauf l'identifiant de ligne
colonnes_med = [c for c in medicaments.columns if c != "id_ligne"]

controles = pd.Series({
    "Âge > 105 ans": (assures["age"] > 105).sum(),
    "Sexe manquant": assures["sexe"].isna().sum(),
    "Canton manquant": assures["canton"].isna().sum(),
    "Prestations : lignes en double": prestations.duplicated(subset=colonnes_prest).sum(),
    "Prestations : montant négatif": (prestations["cout_chf"] < 0).sum(),
    "Prestations : montant nul": (prestations["cout_chf"] == 0).sum(),
    f"Prestations : date hors {ANNEE}": (prestations["date_prestation"].dt.year != ANNEE).sum(),
    "Médicaments : lignes en double": medicaments.duplicated(subset=colonnes_med).sum(),
    "Médicaments : code ATC mal écrit": (medicaments["code_atc"] != medicaments["code_atc"].str.strip().str.upper()).sum(),
}, name="Nombre")
controles.to_frame()

In [ ]:
# Exemple : lignes d'une même personne, même date et même type de prestation qu'une ligne à montant négatif
cas = prestations[prestations["cout_chf"] < 0].iloc[0]
prestations[(prestations["id_assure"] == cas["id_assure"])
            & (prestations["date_prestation"] == cas["date_prestation"])
            & (prestations["type_prestation"] == cas["type_prestation"])]

**Règles de préparation retenues.** On ne touche pas aux tables d'origine, les analyses portent sur des copies. Les âges supérieurs à 105 ans sont considérés comme manquants, les doublons exacts sont supprimés, les prestations antérieures à 2024 sont exclues et les codes ATC sont normalisés. Les lignes d'annulation sont conservées dans le calcul des montants, puisqu'elles compensent la facture d'origine. Pour dénombrer les recours, les lignes d'une même personne, à une même date, pour un même type de prestation et une même cause, sont regroupées en un **contact** lorsque leur montant net est positif.

In [ ]:
# Assuré·es
assures_ok = assures.copy()
assures_ok.loc[assures_ok["age"] > 105, "age"] = np.nan
assures_ok["classe_age"] = pd.cut(assures_ok["age"], bins=[18, 39, 49, 59, 69, 79, 110],
                                  labels=["19-39", "40-49", "50-59", "60-69", "70-79", "80+"])
assures_ok["classe_ofsp"] = pd.cut(assures_ok["age"], bins=CLASSES_OFSP, right=False, labels=LIBELLES_OFSP)

# Médicaments
med_ok = medicaments.drop_duplicates(subset=colonnes_med).copy()
med_ok["code_atc"] = med_ok["code_atc"].str.strip().str.upper()

# Prestations
prest_ok = prestations.drop_duplicates(subset=colonnes_prest)
prest_ok = prest_ok[prest_ok["date_prestation"].dt.year == ANNEE].copy()

# Contacts : une personne, un jour, un type de prestation, une cause, montant net positif
contacts = (prest_ok
            .groupby(["id_assure", "date_prestation", "type_prestation", "cause"], as_index=False)
            .agg(cout_chf=("cout_chf", "sum"), specialite=("specialite", "first")))
contacts = contacts[contacts["cout_chf"] > 0]

print(f"Prestations : {len(prestations):,} lignes → {len(prest_ok):,} lignes nettoyées → {len(contacts):,} contacts")
print(f"Médicaments : {len(medicaments):,} lignes → {len(med_ok):,} délivrances")

**Exercice 1. Qui facture les analyses de laboratoire ?** *(≈ 5 min)*

La variable `type_prestation` décrit la nature de la prestation, la variable `categorie_prestataire` le type de prestataire qui la facture. Calculer la répartition (en %) des lignes de laboratoire selon la catégorie de prestataire.

In [ ]:
## Exercice 1 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# labo = prest_ok[prest_ok["type_prestation"] == ___]
# labo["categorie_prestataire"].value_counts(normalize=___) * 100

<details><summary><b>Indice</b></summary>

Les valeurs possibles de `type_prestation` sont dans le dictionnaire des données. Une valeur textuelle s'écrit entre guillemets. L'option `normalize` de `value_counts()` vaut `True` ou `False` : l'une donne des effectifs, l'autre des proportions.

</details>

**Questions (exercice 1)**
1. Pourquoi une analyse de laboratoire peut-elle être facturée par un cabinet médical ?
2. Qu'implique cette distinction entre type de prestation et catégorie de prestataire lorsqu'on compare des coûts « par prestataire » ?

**Questions**
1. Quelle est l'unité d'observation de chacune des trois tables principales ?
2. Quelle information, nécessaire à l'étude de l'hypertension, ces données ne contiennent-elles pas ?
3. Comment interpréter la succession de lignes de l'exemple ci-dessus ? Quelles conséquences aurait un dénombrement des lignes pour mesurer le nombre de consultations ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 2. Qui sont les assuré·es ?

L'âge, le sexe et les choix d'assurance pèsent lourd sur les dépenses. On les décrit avant toute comparaison.

In [ ]:
# Pyramide des âges par classes de 5 ans (la première va de 19 à 24 ans)
bornes = [19] + list(range(25, 100, 5)) + [120]
noms = [f"{a}-{b - 1}" for a, b in zip(bornes[:-2], bornes[1:-1])] + ["95+"]
pyramide = (assures_ok.dropna(subset=["age", "sexe"])
            .assign(classe=lambda d: pd.cut(d["age"], bornes, right=False, labels=noms))
            .groupby(["classe", "sexe"], observed=False).size().unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(7, 5))
y = np.arange(len(pyramide))
ax.barh(y, -pyramide["M"], height=0.85, color=COULEURS_SEXE["M"], label="Hommes")
ax.barh(y, pyramide["F"], height=0.85, color=COULEURS_SEXE["F"], label="Femmes")
ax.set_yticks(y, pyramide.index)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{abs(v):,.0f}".replace(",", "'")))
ax.axvline(0, color="black", lw=0.8)
ax.legend(loc="upper right")
ax.set(title="Pyramide des âges des assuré·es", xlabel="Nombre de personnes", ylabel="Âge (ans)");

In [ ]:
# Franchise choisie selon l'âge (franchises intermédiaires regroupées)
GROUPES_FRANCHISE = {300: "300", 500: "500", 1000: "1000-2000", 1500: "1000-2000", 2000: "1000-2000", 2500: "2500"}
franchise = pd.crosstab(assures_ok["classe_ofsp"], assures_ok["franchise_chf"].map(GROUPES_FRANCHISE),
                        normalize="index")[["300", "500", "1000-2000", "2500"]]
ax = franchise.plot.barh(stacked=True, color=RAMPE, width=0.8, edgecolor="white", linewidth=1, figsize=(8, 4))
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(title="Franchise (CHF)", bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Franchise choisie selon l'âge", xlabel="Part des assuré·es", ylabel="Classe d'âge");

**Exercice 2. Franchise maximale selon le sexe** *(≈ 7 min)*

Selon les données individuelles des assureurs publiées par l'OFSP (EFIND 2024), la franchise de 2 500 CHF est choisie par 68 % des hommes et 54 % des femmes de 26 à 35 ans, et par 34 % des hommes et 25 % des femmes de 56 à 65 ans. Calculer, par classe d'âge, la part (en %) des femmes et des hommes ayant choisi cette franchise.

In [ ]:
## Exercice 2 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# franchise_max = assures_ok["franchise_chf"] == ___
# part_2500 = 100 * franchise_max.groupby([assures_ok["classe_ofsp"], assures_ok[___]], observed=True).mean().unstack()
# part_2500

<details><summary><b>Indice</b></summary>

La comparaison `==` donne vrai ou faux pour chaque personne. La moyenne de ces valeurs dans un groupe est une proportion. Le second critère de regroupement est le sexe : reprendre le nom de la colonne correspondante, entre guillemets.

</details>

**Questions (exercice 2)**
1. À âge égal, les femmes choisissent-elles moins souvent la franchise maximale que les hommes ? À quels âges l'écart est-il le plus marqué ?
2. Quelles explications proposer ?

In [ ]:
# Modèle d'assurance selon le canton
MODELES = ["standard", "medecin_de_famille", "hmo", "telmed"]
modeles = pd.crosstab(assures_ok["canton"], assures_ok["modele_assurance"], normalize="index")[MODELES]
ax = modeles.rename(columns=libelle).plot.barh(stacked=True, color=CATEGORIELLES[:4], width=0.8, edgecolor="white",
                                           linewidth=1, figsize=(8, 3.5))
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(title="Modèle", bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Modèle d'assurance selon le canton", xlabel="Part des assuré·es", ylabel="");

**Exercice 3. Subsides par canton** *(≈ 7 min, facultatif)*

Selon la statistique de l'assurance-maladie obligatoire (OFSP, 2024, tableau T 4.02, tous âges), le taux de bénéficiaires d'une réduction individuelle de primes atteint 43,9 % à Genève, 36,8 % dans le canton de Vaud, 28,5 % dans le Jura, 27,9 % à Fribourg, 26,6 % à Berne, 23,7 % en Valais et 22,3 % à Neuchâtel. Calculer ce taux (en %) par canton dans les données.

In [ ]:
## Exercice 3 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# taux_subside = 100 * assures_ok.groupby(___)["subside"].mean()
# taux_subside.sort_values(ascending=False)

<details><summary><b>Indice</b></summary>

Grouper par la colonne qui contient le canton de domicile. La variable `subside` vaut 1 pour les bénéficiaires et 0 pour les autres : sa moyenne est une proportion.

</details>

**Questions (exercice 3)**
1. Le classement des cantons est-il le même que dans la statistique fédérale ? Pourquoi les niveaux peuvent-ils différer dans les données d'un assureur ?
2. Les écarts entre cantons reflètent-ils des différences de revenu, de politique cantonale, ou les deux ?

**Questions**
1. Comment les assuré·es se répartissent-ils selon l'âge et le sexe ? Que se passe-t-il au-delà de 80 ans ?
2. Comment le choix de la franchise évolue-t-il avec l'âge ? Pourquoi les personnes âgées choisissent-elles presque toutes la franchise minimale ? Quelle particularité présentent les 19 à 25 ans ?
3. Les modèles d'assurance diffèrent-ils d'un canton à l'autre ? Quelle hypothèse pour le Valais ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 3. Qui coûte, et pourquoi ?

Le coût est défini comme le montant décompté à la charge de l'assurance obligatoire des soins en 2024, prestations et médicaments confondus. Pour les séjours hospitaliers, seule la part payée par l'assureur (45 %) figure dans les données. La part cantonale (55 %) n'y est pas. Les médicaments, dont la cause n'est pas indiquée, sont rattachés à la maladie.

In [ ]:
# Coût par personne et par cause
par_cause = (prest_ok.pivot_table(index="id_assure", columns="cause", values="cout_chf", aggfunc="sum")
             .reindex(columns=["maladie", "maternite", "accident"]))
couts_cause = par_cause.reindex(assures_ok["id_assure"]).fillna(0)
couts_cause["maladie"] += med_ok.groupby("id_assure")["cout_chf"].sum().reindex(couts_cause.index).fillna(0)
couts_cause["total"] = couts_cause[["maladie", "maternite", "accident"]].sum(axis=1)
couts_cause = couts_cause.join(assures_ok.set_index("id_assure")[["sexe", "age", "classe_ofsp", "couverture_accident"]])

(couts_cause[["maladie", "maternite", "accident", "total"]].sum() / couts_cause["total"].sum()).map("{:.1%}".format).to_frame("Part des coûts")

### 3.1 Dépenses selon l'âge et le sexe

In [ ]:
moyennes_sexe = couts_cause.groupby(["classe_ofsp", "sexe"], observed=True)["total"].mean().unstack()
fig, ax = plt.subplots()
for sexe, nom in [("F", "Femmes"), ("M", "Hommes")]:
    ax.plot(moyennes_sexe.index.astype(str), moyennes_sexe[sexe], marker="o", ms=6, lw=2, color=COULEURS_SEXE[sexe], label=nom)
axe_chf(ax)
ax.legend()
ax.set(title="Coût moyen par personne selon l'âge et le sexe", xlabel="Classe d'âge", ylabel="CHF");

### 3.2 Pourquoi les femmes de 20 à 45 ans coûtent-elles davantage ?

La variable `cause` distingue les prestations liées à la **maladie**, à la **maternité** (suivi de grossesse, accouchement, suites de couches) et aux **accidents**.

In [ ]:
par_cause_age = couts_cause.groupby(["sexe", "classe_ofsp"], observed=True)[["maladie", "maternite", "accident"]].mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (sexe, nom) in zip(axes, [("F", "Femmes"), ("M", "Hommes")]):
    par_cause_age.loc[sexe].plot.bar(stacked=True, ax=ax, width=0.8, edgecolor="white", linewidth=1, legend=False,
                                     color=[COULEURS_CAUSE[c] for c in ["maladie", "maternite", "accident"]])
    ax.set(title=nom, xlabel="Classe d'âge", ylabel="CHF par personne")
    ax.tick_params(axis="x", rotation=0)
axe_chf(axes[0])
axes[1].legend(["Maladie", "Maternité", "Accident"], bbox_to_anchor=(1, 1), loc="upper left")
fig.suptitle("Coût moyen par personne selon la cause", fontweight="bold");

In [ ]:
# Écart entre femmes et hommes de 26 à 35 ans, et part expliquée par la maternité
jeunes = couts_cause[couts_cause["age"].between(26, 35)].groupby("sexe")[["maladie", "maternite", "accident", "total"]].mean()
ecart = jeunes.loc["F", "total"] - jeunes.loc["M", "total"]
print(f"Coût moyen à 26-35 ans : femmes CHF {chf(jeunes.loc['F', 'total'])}, hommes CHF {chf(jeunes.loc['M', 'total'])}")
print(f"Écart : CHF {chf(ecart)}, dont maternité : CHF {chf(jeunes.loc['F', 'maternite'])} "
      f"({jeunes.loc['F', 'maternite'] / ecart:.0%} de l'écart)")
jeunes.T

**Exercice 4. La part payée par les assuré·es** *(≈ 8 min)*

Hormis la maternité, les prestations donnent lieu à une participation des assuré·es (franchise, quote-part de 10 %, contribution aux frais de séjour hospitalier) : c'est la colonne `participation_chf`. Selon la statistique de l'OFSP (2024, adultes de 26 ans et plus, calcul à partir des tableaux T 2.23 et T 2.24), cette participation représente 7 à 8 % des coûts avec une franchise de 300 CHF et 41 à 49 % avec une franchise de 2 500 CHF, selon le modèle d'assurance. Calculer cette part (en %) selon la franchise.

In [ ]:
## Exercice 4 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# lignes = pd.concat([prest_ok, med_ok])[["id_assure", "cout_chf", "participation_chf"]]
# lignes = lignes.merge(assures_ok[["id_assure", "franchise_chf"]], on="id_assure")
# par_franchise = lignes.groupby(___)[["cout_chf", "participation_chf"]].sum()
# par_franchise["pct_paye"] = 100 * par_franchise[___] / par_franchise[___]
# par_franchise

<details><summary><b>Indice</b></summary>

Grouper par la colonne de la franchise. La part payée rapporte la somme des participations à la somme des coûts : quelle colonne au numérateur, laquelle au dénominateur ?

</details>

**Questions (exercice 4)**
1. Quelle part de leurs coûts les assuré·es paient-ils eux-mêmes avec une franchise de 300 CHF ? Et de 2 500 CHF ?
2. Pourquoi cette part est-elle probablement sous-estimée dans les données de l'assureur ?

### 3.3 Les accidents

L'assurance obligatoire des soins ne couvre l'accident que pour les personnes qui ne sont pas assurées contre les accidents par leur employeur (LAA) : les salarié·es travaillant au moins 8 heures par semaine sont couvert·es par la LAA, et leurs accidents n'apparaissent pas dans ces données. La variable `couverture_accident` indique les personnes couvertes par la LAMal.

In [ ]:
accidents = couts_cause.groupby("classe_ofsp", observed=True).agg(couverture=("couverture_accident", "mean"),
                                                                   cout_accident=("accident", "mean"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
accidents["couverture"].plot.bar(ax=axes[0], color="#2A78D6", width=0.75)
axe_pct(axes[0])
axes[0].set(title="Couvert·es contre les accidents par la LAMal", xlabel="Classe d'âge", ylabel="Part des assuré·es")
accidents["cout_accident"].plot.bar(ax=axes[1], color=COULEURS_CAUSE["accident"], width=0.75)
axe_chf(axes[1])
axes[1].set(title="Coût moyen des accidents par personne", xlabel="Classe d'âge", ylabel="CHF")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)

**Exercice 5. Accidents des personnes âgées** *(≈ 5 min, facultatif)*

Parmi les personnes de 76 ans et plus couvertes contre les accidents par la LAMal, comparer le coût moyen des accidents des femmes et des hommes.

In [ ]:
## Exercice 5 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# ages_couverts = couts_cause[(couts_cause["age"] >= 76) & (couts_cause[___] == 1)]
# ages_couverts.groupby(___)["accident"].agg(["size", "mean"])

<details><summary><b>Indice</b></summary>

`couts_cause.columns` affiche les colonnes disponibles. La couverture contre les accidents vaut 1 pour les personnes couvertes par la LAMal. Grouper ensuite par sexe.

</details>

**Questions (exercice 5)**
1. Le coût moyen des accidents diffère-t-il entre femmes et hommes âgés ? Quelles explications proposer ?
2. Sur combien de personnes accidentées ce résultat repose-t-il ? (`(ages_couverts["accident"] > 0).sum()`)

**Questions**
1. À quels âges les dépenses des femmes dépassent-elles celles des hommes ? Quelle part de l'écart la maternité explique-t-elle à 26-35 ans, et quelles hypothèses avancer pour le reste ?
2. Les prestations de maternité ne donnent lieu à aucune participation (franchise, quote-part). Quelle est la justification de cette règle ?
3. Pourquoi les accidents apparaissent-ils surtout après 65 ans dans ces données ?
4. Pour comparer des personnes atteintes d'une maladie chronique aux autres assuré·es, faut-il exclure les prestations de cause « accident » ? Et celles de cause « maternité » ? Justifier.

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 3.4 Concentration des dépenses

La courbe suivante classe les assuré·es du plus coûteux au moins coûteux et cumule leurs dépenses.

In [ ]:
tri = np.sort(couts_cause["total"].to_numpy())[::-1]
cumul = tri.cumsum() / tri.sum()
part_personnes = np.arange(1, len(tri) + 1) / len(tri)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(part_personnes, cumul, color="#2A78D6", lw=2)
ax.plot([0, 1], [0, 1], color="#7F8C8D", ls=":", lw=1)
for p in (0.01, 0.10, 0.20, 0.50):
    valeur = cumul[int(p * len(tri)) - 1]
    ax.plot(p, valeur, "o", ms=7, color="#2A78D6")
    ax.annotate(f"{p:.0%} des assuré·es : {valeur:.0%} des coûts", (p, valeur), xytext=(10, -14), textcoords="offset points")
axe_pct(ax)
axe_pct(ax, "x")
ax.set(title="Concentration des dépenses", xlabel="Assuré·es, du plus au moins coûteux", ylabel="Part cumulée des dépenses");

In [ ]:
def part_des_plus_couteux(depenses, p):
    """Part des dépenses totales portée par la fraction p des personnes les plus coûteuses."""
    valeurs = np.sort(np.asarray(depenses))[::-1]
    return valeurs[: int(round(p * len(valeurs)))].sum() / valeurs.sum()


concentration = pd.Series({
    "1 % les plus coûteux": part_des_plus_couteux(couts_cause["total"], 0.01),
    "5 % les plus coûteux": part_des_plus_couteux(couts_cause["total"], 0.05),
    "10 % les plus coûteux": part_des_plus_couteux(couts_cause["total"], 0.10),
    "20 % les plus coûteux": part_des_plus_couteux(couts_cause["total"], 0.20),
    "50 % les moins coûteux": 1 - part_des_plus_couteux(couts_cause["total"], 0.50),
}).map("{:.1%}".format).to_frame("Part des dépenses")
print(f"Assuré·es sans aucun décompte en 2024 : {(couts_cause['total'] <= 0).mean():.1%}")
concentration

In [ ]:
# Qui sont les 10 % les plus coûteux ?
seuil = couts_cause["total"].quantile(0.9)
hospitalises = contacts.loc[contacts["type_prestation"] == "stationnaire", "id_assure"].unique()
deces = assures_ok.loc[assures_ok["motif_fin_couverture"] == "deces", "id_assure"]
substances = med_ok.groupby("id_assure")["code_atc"].nunique()

profil_couts = couts_cause.assign(
    groupe=np.where(couts_cause["total"] >= seuil, "10 % les plus coûteux", "90 % restants"),
    hospitalise=couts_cause.index.isin(hospitalises),
    decede=couts_cause.index.isin(deces),
    substances=substances.reindex(couts_cause.index).fillna(0),
)
profil_couts.groupby("groupe").agg(
    personnes=("total", "size"),
    age_moyen=("age", "mean"),
    pct_femmes=("sexe", lambda s: 100 * (s == "F").mean()),
    pct_hospitalises=("hospitalise", lambda s: 100 * s.mean()),
    pct_decedes=("decede", lambda s: 100 * s.mean()),
    substances_differentes=("substances", "mean"),
    cout_moyen=("total", "mean"),
).T

**Exercice 6. Concentration chez les jeunes et chez les âgé·es** *(≈ 8 min, facultatif)*

La fonction `part_des_plus_couteux(depenses, p)`, définie plus haut, calcule la part des dépenses portée par la fraction `p` des personnes les plus coûteuses. Comparer la part des dépenses portée par les 10 % les plus coûteux chez les 19 à 35 ans et chez les 66 ans et plus.

In [ ]:
## Exercice 6 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# depenses_jeunes = couts_cause.loc[couts_cause["age"] < 36, "total"]
# depenses_agees = couts_cause.loc[couts_cause["age"] >= ___, "total"]
# print(f"19-35 ans : les 10 % les plus coûteux portent {part_des_plus_couteux(depenses_jeunes, ___):.0%} des dépenses")
# print(f"66 ans et plus : les 10 % les plus coûteux portent {part_des_plus_couteux(___, 0.10):.0%} des dépenses")

<details><summary><b>Indice</b></summary>

Le second groupe commence à 66 ans. Une fraction de 10 % s'écrit `0.10`. Le second appel de la fonction porte sur les dépenses du second groupe.

</details>

**Questions (exercice 6)**
1. Dans quel groupe d'âge les dépenses sont-elles le plus concentrées ? Pourquoi ?

**Questions**
1. Les dépenses suivent-elles la « règle des 80/20 » ? Quelle part des dépenses représentent les 1 % les plus coûteux, et la moitié la moins coûteuse des assuré·es ?
2. Pourquoi une part importante des assuré·es n'a-t-elle aucun décompte dans l'année ?
3. Qu'est-ce qui distingue les 10 % les plus coûteux ? Quelles conséquences ce degré de concentration a-t-il pour le financement de l'assurance obligatoire (solidarité, compensation des risques entre assureurs) ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 3.5 Comparer des coûts : deux pièges d'interprétation

Avec des données de remboursement, on est vite tenté de comparer des coûts entre groupes d'assuré·es ou entre prestataires. Les deux pièges suivants montrent chacun un résultat qui semble parler de lui-même.

**Piège d'interprétation A. La franchise élevée fait-elle baisser les coûts ?** *(≈ 8 min)*

La franchise est souvent présentée comme un levier de maîtrise des coûts. Le graphique compare, selon la franchise choisie, le coût moyen par personne et la part des personnes sans aucun décompte.

In [ ]:
# Coût moyen et part des personnes sans aucun décompte, selon la franchise
selon_franchise = (couts_cause.join(assures_ok.set_index("id_assure")["franchise_chf"])
                   .groupby("franchise_chf")["total"].agg(cout_moyen="mean", sans_decompte=lambda s: (s <= 0).mean()))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
selon_franchise["cout_moyen"].plot.bar(ax=axes[0], color="#2A78D6", width=0.7)
selon_franchise["sans_decompte"].plot.bar(ax=axes[1], color="#7F8C8D", width=0.7)
axe_chf(axes[0])
axe_pct(axes[1])
axes[0].set(title="Coût moyen par personne", xlabel="Franchise (CHF)", ylabel="CHF")
axes[1].set(title="Personnes sans aucun décompte", xlabel="Franchise (CHF)", ylabel="Part des assuré·es")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)

> **Conclusion tentante.** « Avec une franchise de 2 500 CHF, les assuré·es coûtent cinq fois moins qu'avec une franchise de 300 CHF, et un tiers d'entre eux n'ont eu besoin d'aucun soin. Relever la franchise minimale réduirait donc fortement les dépenses de l'assurance obligatoire. »

**Question (piège A).** Pourquoi les données de remboursement ne permettent-elles pas de tirer cette conclusion ? Distinguer ce qui tient au choix des assuré·es, au comportement induit par la franchise et à la manière dont les données sont produites. *Répondre avant d'exécuter la cellule suivante.*

In [ ]:
# Vérification : qui choisit quelle franchise ?
(profil_couts.join(assures_ok.set_index("id_assure")["franchise_chf"])
 .groupby("franchise_chf")
 .agg(personnes=("total", "size"), age_moyen=("age", "mean"),
      pct_hospitalises=("hospitalise", lambda s: 100 * s.mean()),
      substances_differentes=("substances", "mean")))

**Question (piège A, suite).** Que montre ce tableau ? Pourquoi la part des personnes hospitalisées renseigne-t-elle mieux sur l'état de santé que le nombre de consultations ? Le même raisonnement s'applique-t-il aux modèles d'assurance alternatifs (médecin de famille, HMO, télémédecine), dont les assuré·es coûtent eux aussi beaucoup moins que ceux du modèle standard ?

**Piège d'interprétation B. Peut-on repérer les médecins dépensiers ?** *(≈ 8 min)*

Les données indiquent le prestataire de chaque prestation. Chaque assuré·e est rattaché·e au cabinet de médecine générale qu'il ou elle a consulté le plus souvent en 2024. Le graphique compare le coût moyen des patient·es (prestations et médicaments) des cabinets qui en comptent au moins huit dans les données.

In [ ]:
# Rattachement de chaque assuré·e au cabinet de médecine générale le plus consulté en 2024
jours = (prest_ok[(prest_ok["type_prestation"] == "medecin_generaliste") & (prest_ok["cout_chf"] > 0)]
         .groupby(["id_assure", "id_prestataire"])["date_prestation"].nunique().reset_index())
cabinet = jours.loc[jours.groupby("id_assure")["date_prestation"].idxmax()].set_index("id_assure")["id_prestataire"]
patients = profil_couts.join(cabinet.rename("cabinet"), how="inner")

# Coût moyen par patient·e des cabinets comptant au moins 8 patient·es
cabinets = patients.groupby("cabinet")["total"].agg(["size", "mean"])
cabinets = cabinets[cabinets["size"] >= 8].sort_values("mean")
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(cabinets.index[:10], cabinets["mean"].iloc[:10], height=0.7, color="#2A78D6", label="10 cabinets les moins coûteux")
ax.barh(cabinets.index[-10:], cabinets["mean"].iloc[-10:], height=0.7, color="#EB6834", label="10 cabinets les plus coûteux")
ax.axvline(patients["total"].mean(), color="black", ls="--", lw=1, label="Moyenne de l'ensemble des patient·es")
axe_chf(ax, "x")
ax.legend(loc="lower right")
ax.set(title="Coût moyen par patient·e, selon le cabinet de médecine générale", xlabel="CHF par patient·e", ylabel="");

> **Conclusion tentante.** « Les patient·es des cabinets les plus coûteux coûtent jusqu'à trente fois plus que ceux des cabinets les moins coûteux. L'assureur dispose ainsi de la liste des médecins dépensiers, auxquels il peut demander de justifier leurs coûts. »

**Question (piège B).** Les données permettent-elles d'identifier les cabinets coûteux ? Permettent-elles d'expliquer pourquoi ils le sont ? Proposer des explications autres que la pratique des médecins. *Répondre avant d'exécuter la cellule suivante.*

In [ ]:
# Vérification : patientèle des 20 cabinets du graphique
print(f"{len(cabinet):,} personnes rattachées à {cabinet.nunique():,} cabinets, "
      f"soit {len(cabinet) / cabinet.nunique():.1f} patient·es par cabinet en moyenne")
extremes = patients[patients["cabinet"].isin(cabinets.index[:10].append(cabinets.index[-10:]))]
(extremes.assign(groupe=np.where(extremes["cabinet"].isin(cabinets.index[-10:]), "10 plus coûteux", "10 moins coûteux"))
 .groupby(["groupe", "cabinet"])
 .agg(patients=("total", "size"), age_moyen=("age", "mean"), hospitalises=("hospitalise", "sum"),
      pct_patient_le_plus_couteux=("total", lambda s: 100 * s.max() / s.sum()))
 .groupby("groupe").mean())

**Question (piège B, suite).** Que montre ce tableau ? Combien de patient·es par cabinet un assureur de 15 000 assuré·es observe-t-il ? De quelles données et de quelles méthodes faudrait-il disposer pour comparer équitablement des médecins ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 4. Personnes traitées pour hypertension

Les sections précédentes ont montré que les dépenses augmentent avec l'âge et se concentrent sur une minorité de personnes, souvent atteintes de maladies chroniques. Cette section étudie l'une d'elles, l'hypertension artérielle. Comme discuté en section 3, le recours aux soins et les coûts sont calculés sur les prestations de cause « maladie » et sur les médicaments : les accidents et la maternité, dont l'observation dépend de la couverture et du sexe, en sont exclus.

### 4.1 Identification des personnes traitées

Les données de facturation ambulatoire ne comportent pas de diagnostic. Les personnes traitées sont donc identifiées à partir des médicaments délivrés, selon la définition opérationnelle suivante : **au moins deux délivrances en 2024 d'un antihypertenseur**, c'est-à-dire d'un médicament dont le code ATC commence par l'un des préfixes ci-dessous. Les personnes ainsi identifiées forment le groupe « HTA traitée » (hypertension artérielle traitée).

| C02 | C03 | C07 | C08 | C09 |
|---|---|---|---|---|
| action centrale / alpha-bloquants | diurétiques | bêtabloquants | inhibiteurs calciques | IEC / sartans (ARA) |

In [ ]:
# Paramètres de la définition opérationnelle (modifiables, l'exercice 7 teste le seuil)
CLASSES_ANTIHYPERTENSEURS = ["C02", "C03", "C07", "C08", "C09"]
SEUIL_DELIVRANCES = 2

In [ ]:
antihta = med_ok[med_ok["code_atc"].str[:3].isin(CLASSES_ANTIHYPERTENSEURS)]
nb_delivrances = antihta.groupby("id_assure").size()
ids_hta = nb_delivrances[nb_delivrances >= SEUIL_DELIVRANCES].index

assures_ok["groupe"] = np.where(assures_ok["id_assure"].isin(ids_hta), "HTA traitée", "Autres assuré·es")
hta = assures_ok["groupe"] == "HTA traitée"

print(f"{hta.sum():,} personnes traitées pour hypertension, soit {hta.mean():.1%} des assuré·es")

In [ ]:
ax = nb_delivrances.clip(upper=20).value_counts().sort_index().plot.bar(color="#34495E", width=0.8)
seuil_retenu = ax.axvline(SEUIL_DELIVRANCES - 1.5, color="#C0392B", ls="--", label=f"Seuil retenu : {SEUIL_DELIVRANCES} délivrances")
ax.legend(handles=[seuil_retenu])
ax.set(title="Nombre de délivrances d'antihypertenseurs par personne (20 = 20 ou plus)",
       xlabel="Délivrances en 2024", ylabel="Nombre de personnes")
ax.tick_params(axis="x", rotation=0);

Les bêtabloquants et les diurétiques ont d'autres indications que l'hypertension. Le tableau suivant décrit les personnes identifiées sur la base d'une seule classe thérapeutique, bêtabloquant (C07) ou diurétique de l'anse (C03C).

In [ ]:
classes_par_personne = antihta.groupby("id_assure")["code_atc"].agg(lambda s: set(s.str[:3]))
une_seule_classe = classes_par_personne[classes_par_personne.apply(len) == 1].index.intersection(ids_hta)

a_verifier = (antihta[antihta["id_assure"].isin(une_seule_classe)
                      & antihta["code_atc"].str.startswith(("C07", "C03C"))]
              .drop_duplicates(["id_assure", "code_atc"])
              .merge(referentiel[["code_atc", "substance"]], on="code_atc")
              .merge(assures_ok[["id_assure", "age", "sexe"]], on="id_assure"))

anticoagules = med_ok.loc[med_ok["code_atc"].str.startswith("B01AF"), "id_assure"].unique()
a_verifier["aussi_anticoagulant"] = a_verifier["id_assure"].isin(anticoagules)

(a_verifier.groupby("substance")
 .agg(personnes=("id_assure", "nunique"), age_median=("age", "median"),
      pct_femmes=("sexe", lambda s: 100 * (s == "F").mean()),
      pct_avec_anticoagulant=("aussi_anticoagulant", lambda s: 100 * s.mean()))
 .sort_values("personnes", ascending=False))

**Exercice 7. Sensibilité de la définition** *(≈ 6 min)*

Compter les personnes identifiées avec un seuil de 1, 2, 4 ou 6 délivrances d'antihypertenseurs dans l'année.

In [ ]:
## Exercice 7 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# for seuil in [1, 2, 4, 6]:
#     n = (nb_delivrances >= ___).sum()
#     print(f"Au moins {seuil} délivrance(s) : {n:,} personnes ({n / len(assures_ok):.1%} des assuré·es)")

<details><summary><b>Indice</b></summary>

À chaque passage de la boucle, la variable `seuil` prend l'une des valeurs de la liste. `nb_delivrances` contient le nombre de délivrances de chaque personne.

</details>

**Questions (exercice 7)**
1. Comment la proportion de personnes identifiées varie-t-elle avec le seuil ?
2. Les traitements au long cours sont souvent délivrés en boîtes de 98 ou 100 comprimés. Combien de délivrances par an une personne traitée toute l'année par un seul médicament reçoit-elle alors ? Quelle conséquence pour le choix du seuil ?

**Questions**
1. La proportion de personnes identifiées est-elle plausible ?
2. Quelles personnes cette définition inclut-elle à tort ? On pourra considérer l'âge et le sexe des personnes traitées par propranolol, ainsi que l'association d'un bêtabloquant et d'un anticoagulant.

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.2 Caractéristiques des personnes traitées

Les autres maladies chroniques sont, elles aussi, approchées par les médicaments délivrés : antidiabétiques (A10), statines (C10) et anticoagulants oraux directs (B01AF).

In [ ]:
prevalence = assures_ok.groupby(["classe_age", "sexe"], observed=True)["groupe"].apply(lambda g: (g == "HTA traitée").mean())
ax = prevalence.reset_index(name="part").pipe(
    lambda d: sns.barplot(data=d, x="classe_age", y="part", hue="sexe", palette=COULEURS_SEXE, hue_order=["F", "M"], saturation=1))
axe_pct(ax)
ax.legend(ax.get_legend_handles_labels()[0], ["Femmes", "Hommes"])
ax.set(title="Part des personnes traitées pour hypertension", xlabel="Classe d'âge", ylabel="");

**Piège d'interprétation C. Que mesure l'indicateur « HTA traitée » ?** *(≈ 10 min)*

Le graphique compare la part des personnes identifiées comme traitées pour hypertension dans les données à la part des personnes qui déclarent une hypertension dans l'Enquête suisse sur la santé (ESS 2022, Obsan, où la première classe d'âge va de 15 à 24 ans).

In [ ]:
# Hypertension traitée (données) et hypertension déclarée (Enquête suisse sur la santé 2022, en %)
ESS_HTA = pd.DataFrame({"F": [0.7, 1.7, 3.2, 8.8, 21.0, 35.7, 54.9], "M": [2.4, 3.8, 7.6, 18.0, 34.2, 47.6, 59.4]},
                       index=["19-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"]) / 100
classe_ess = pd.cut(assures_ok["age"], [18, 24, 34, 44, 54, 64, 74, 120], labels=ESS_HTA.index)
hta_donnees = hta.groupby([classe_ess, assures_ok["sexe"]], observed=True).mean().unstack()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (sexe, nom) in zip(axes, [("F", "Femmes"), ("M", "Hommes")]):
    ax.bar(ESS_HTA.index, hta_donnees[sexe], width=0.7, color=COULEURS_SEXE[sexe], label="Données : traitées (≥ 2 délivrances)")
    ax.plot(ESS_HTA.index, ESS_HTA[sexe], "D", ms=7, color="black", label="ESS 2022 : hypertension déclarée")
    ax.set(title=nom, xlabel="Classe d'âge")
axe_pct(axes[0])
axes[0].set_ylabel("Part des personnes")
axes[1].legend(loc="upper left")
plt.show()
print("Rapport entre les données et l'enquête :")
(hta_donnees / ESS_HTA).T.round(1)

> **Conclusion tentante.** « Chez les femmes de moins de 45 ans, les données comptent nettement plus de personnes traitées pour hypertension que l'enquête ne compte de personnes hypertendues (trois fois plus entre 19 et 24 ans), alors que chez les hommes les deux sources concordent : l'hypertension des jeunes femmes est sous-estimée par les enquêtes. Après 55 ans, les deux sources concordent : environ une personne sur deux est hypertendue à partir de 75 ans, et le groupe « Autres assuré·es » rassemble les personnes qui ne le sont pas. »

**Question (piège C).** Pourquoi les données de remboursement ne permettent-elles pas de tirer ces conclusions ? Que mesure exactement l'indicateur « HTA traitée » ? *Répondre avant d'exécuter la cellule suivante.*

In [ ]:
# Vérification 1 : sans les personnes identifiées uniquement par un bêtabloquant (C07), dont l'indication est inconnue
bb_seul = classes_par_personne[classes_par_personne.apply(lambda c: c == {"C07"})].index.intersection(ids_hta)
sans_bb = (hta & ~assures_ok["id_assure"].isin(bb_seul)).groupby([classe_ess, assures_ok["sexe"]], observed=True).mean().unstack()
print("Femmes, en % :")
display(pd.DataFrame({"Données": hta_donnees["F"], "Données sans « bêtabloquant seul »": sans_bb["F"],
                      "ESS 2022": ESS_HTA["F"]}).mul(100).round(1))

# Vérification 2 : selon l'OMS (2023), 56 % seulement des personnes hypertendues de 30 à 79 ans sont traitées en Suisse
p = hta[assures_ok["age"].between(30, 79)].mean()
print(f"30-79 ans : {p:.0%} de personnes traitées. Si 56 % des hypertendu·es sont traité·es, la prévalence serait d'environ {p / 0.56:.0%}, "
      f"et environ {(p / 0.56 - p) / (1 - p):.0%} des « autres assuré·es » seraient hypertendu·es sans traitement.")

**Question (piège C, suite).** Que montrent ces vérifications ? Quelles données faudrait-il pour mesurer la prévalence de l'hypertension ? Quelle conséquence la présence de personnes hypertendues non traitées dans le groupe « Autres assuré·es » a-t-elle sur les comparaisons de cette section et des suivantes ?

In [ ]:
# Indicateurs de comorbidité fondés sur les délivrances
PROXYS = {"Diabète (A10)": "A10", "Statine (C10)": "C10", "Anticoagulant (B01AF)": "B01AF"}
for nom, prefixe in PROXYS.items():
    assures_ok[nom] = assures_ok["id_assure"].isin(med_ok.loc[med_ok["code_atc"].str.startswith(prefixe), "id_assure"])

profil = assures_ok.groupby("groupe").agg(
    personnes=("id_assure", "size"),
    age_moyen=("age", "mean"),
    pct_femmes=("sexe", lambda s: 100 * (s == "F").mean()),
    pct_subside=("subside", lambda s: 100 * s.mean()),
    **{f"% {nom}": (nom, lambda s: 100 * s.mean()) for nom in PROXYS},
).T[GROUPES]
profil

In [ ]:
comorbidites = assures_ok.groupby("groupe")[list(PROXYS)].mean().T[GROUPES]
ax = comorbidites.plot.barh(color=COULEURS, width=0.75)
ax.invert_yaxis()
ax.legend(title="")
axe_pct(ax, "x")
ax.set(title="Autres traitements chroniques", xlabel="Part des personnes", ylabel="");

**Exercice 8. Statines à âge égal** *(≈ 7 min, facultatif)*

Les personnes traitées pour hypertension sont plus âgées que les autres assuré·es. Comparer, par classe d'âge, la part (en %) de personnes recevant une statine dans les deux groupes.

In [ ]:
## Exercice 8 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# statines_age = 100 * assures_ok.groupby([___, "groupe"], observed=True)["Statine (C10)"].mean().unstack()[GROUPES]
# statines_age

<details><summary><b>Indice</b></summary>

Le premier critère de regroupement est la classe d'âge : reprendre le nom de la colonne utilisée dans les graphiques précédents, entre guillemets.

</details>

**Questions (exercice 8)**
1. L'écart de traitement par statine entre les deux groupes persiste-t-il à âge égal ?
2. Que suggère ce résultat pour la prise en charge des personnes hypertendues ?

La cellule suivante compare la distribution d'une variable entre les deux groupes. Paramètre modifiable : `VARIABLE` (`franchise_chf`, `modele_assurance`, `canton` ou `subside`).

In [ ]:
VARIABLE = "franchise_chf"   # paramètre modifiable

donnees = assures_ok.assign(franchise_chf=assures_ok["franchise_chf"].map(GROUPES_FRANCHISE))
repartition = pd.crosstab(donnees["groupe"], donnees[VARIABLE], normalize="index").loc[GROUPES]
if VARIABLE == "franchise_chf":
    repartition, couleurs = repartition[["300", "500", "1000-2000", "2500"]], RAMPE
else:
    couleurs = CATEGORIELLES[:repartition.shape[1]]
ax = repartition.rename(columns=libelle).plot.barh(stacked=True, color=couleurs, figsize=(8, 2.8), width=0.7,
                                                  edgecolor="white", linewidth=1)
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title=f"{libelle(VARIABLE)} selon le groupe", xlabel="", ylabel="");

**Questions**
1. Comment caractériser, en deux phrases, les personnes traitées pour hypertension ?
2. Comment expliquer la fréquence plus élevée de la franchise minimale dans ce groupe ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.3 Traitements antihypertenseurs

Certains médicaments sont des associations fixes, qui réunissent plusieurs classes thérapeutiques sous un seul code ATC (par exemple `C09DA` : antagoniste de l'angiotensine II et diurétique). Chaque code est donc converti en la ou les classes qu'il contient.

In [ ]:
CLASSES = {   # début du code ATC → classes contenues
    "C09AA": ["IEC"], "C09BA": ["IEC", "Diurétique"], "C09BB": ["IEC", "Inhibiteur calcique"],
    "C09CA": ["Sartan (ARA)"], "C09DA": ["Sartan (ARA)", "Diurétique"],
    "C09DB": ["Sartan (ARA)", "Inhibiteur calcique"],
    "C09DX": ["Sartan (ARA)", "Inhibiteur calcique", "Diurétique"],
    "C08": ["Inhibiteur calcique"], "C07": ["Bêtabloquant"], "C02": ["Autre antihypertenseur"],
    "C03A": ["Diurétique"], "C03B": ["Diurétique"], "C03C": ["Diurétique de l'anse"], "C03D": ["Antagoniste aldostérone"],
}


def classes_du_code(code_atc):
    for debut in sorted(CLASSES, key=len, reverse=True):
        if code_atc.startswith(debut):
            return CLASSES[debut]
    return ["Non classé"]


trt = antihta[antihta["id_assure"].isin(ids_hta)].copy()
trt["classe"] = trt["code_atc"].apply(classes_du_code)
trt["combinaison_fixe"] = trt["classe"].apply(len) > 1
trt_classes = trt.explode("classe")

par_personne = pd.DataFrame({
    "nb_codes": trt.groupby("id_assure")["code_atc"].nunique(),
    "nb_classes": trt_classes.groupby("id_assure")["classe"].nunique(),
    "combinaison_fixe": trt.groupby("id_assure")["combinaison_fixe"].any(),
})
print(f"{par_personne['combinaison_fixe'].mean():.0%} des personnes reçoivent au moins une combinaison fixe")

In [ ]:
part_classes = trt_classes.groupby("classe")["id_assure"].nunique().div(len(par_personne)).sort_values()
ax = part_classes.plot.barh(color=COULEURS["HTA traitée"])
axe_pct(ax, "x")
ax.set(title="Classes utilisées (au moins une fois en 2024)", xlabel="Part des personnes traitées", ylabel="");

In [ ]:
comparaison = pd.DataFrame({
    "Compté en codes ATC": par_personne["nb_codes"].clip(upper=4).value_counts(normalize=True),
    "Compté en classes": par_personne["nb_classes"].clip(upper=4).value_counts(normalize=True),
}).sort_index().rename(index={4: "4+"})
ax = comparaison.plot.bar(color=["#BDC3C7", COULEURS["HTA traitée"]], width=0.75)
axe_pct(ax)
ax.tick_params(axis="x", rotation=0)
ax.set(title="Nombre de médicaments antihypertenseurs par personne", xlabel="Nombre", ylabel="Part des personnes");

In [ ]:
par_personne["classe_age"] = assures_ok.set_index("id_assure")["classe_age"].reindex(par_personne.index)
polytherapie = pd.crosstab(par_personne["classe_age"], par_personne["nb_classes"].clip(upper=3), normalize="index")
ax = polytherapie.rename(columns={1: "1 classe", 2: "2 classes", 3: "3 classes ou +"}).plot.bar(
    stacked=True, color=["#F5B7B1", "#E74C3C", "#922B21"], width=0.75, edgecolor="white", linewidth=1)
axe_pct(ax)
ax.tick_params(axis="x", rotation=0)
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Nombre de classes selon l'âge", xlabel="Classe d'âge", ylabel="");

**Questions**
1. Quelles sont les classes les plus utilisées ? La monothérapie est-elle majoritaire, et comment évolue-t-elle avec l'âge ?
2. Pourquoi un dénombrement des codes ATC sous-estime-t-il la fréquence des associations thérapeutiques ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.4 Recours aux soins

Le recours aux soins est mesuré par le nombre de contacts de cause « maladie » (définis à la section 1) par personne et par type de prestation.

In [ ]:
contacts_maladie = contacts[contacts["cause"] == "maladie"]
recours = (pd.crosstab(contacts_maladie["id_assure"], contacts_maladie["type_prestation"])
           .reindex(assures_ok["id_assure"], fill_value=0))
recours["hospitalise"] = recours["stationnaire"] > 0
recours = recours.join(assures_ok.set_index("id_assure")[["groupe", "classe_age"]])

moyennes = recours.groupby("groupe")[["medecin_generaliste", "medecin_specialiste", "laboratoire",
                                      "ambulatoire_hospitalier", "physiotherapie"]].mean().T[GROUPES]
ax = moyennes.rename(index=libelle).plot.barh(color=COULEURS, width=0.75)
ax.invert_yaxis()
ax.legend(title="")
ax.set(title="Nombre moyen de contacts par personne en 2024", xlabel="Contacts", ylabel="");

In [ ]:
print("Part des personnes hospitalisées au moins une fois :")
recours.groupby("groupe")["hospitalise"].mean().loc[GROUPES].map("{:.1%}".format).to_frame("")

**Piège d'interprétation D. Les associations d'antihypertenseurs sont-elles dangereuses ?** *(≈ 7 min)*

Le graphique compare la part des personnes traitées hospitalisées au moins une fois en 2024, selon le nombre de classes d'antihypertenseurs reçues (section 4.3).

In [ ]:
# Hospitalisation selon le nombre de classes d'antihypertenseurs (personnes traitées)
poly = par_personne.join(recours["hospitalise"])
poly["traitement"] = poly["nb_classes"].clip(upper=3).map({1: "1 classe", 2: "2 classes", 3: "3 classes ou +"})
taux_hospitalisation = poly.groupby("traitement")["hospitalise"].mean()
ax = taux_hospitalisation.plot.bar(color=COULEURS["HTA traitée"], width=0.6, figsize=(6, 3.8))
ax.bar_label(ax.containers[0], labels=[f"{v:.0%}" for v in taux_hospitalisation], padding=3)
ax.set_ylim(0, 1.2 * taux_hospitalisation.max())
axe_pct(ax)
ax.tick_params(axis="x", rotation=0)
ax.set(title="Personnes hospitalisées au moins une fois en 2024", xlabel="Traitement antihypertenseur", ylabel="Part des personnes traitées");

> **Conclusion tentante.** « Plus les personnes reçoivent de classes d'antihypertenseurs, plus elles sont hospitalisées : 14 % avec une seule classe, 26 % avec trois classes ou plus. Les associations d'antihypertenseurs augmentent le risque d'hospitalisation et devraient être limitées. »

**Question (piège D).** Pourquoi ces données ne permettent-elles pas de conclure que les associations d'antihypertenseurs provoquent des hospitalisations ? Proposer d'autres explications. *Répondre avant d'exécuter la cellule suivante.*

In [ ]:
# Vérification : âge et maladies associées selon le nombre de classes
MARQUEURS = {"Anticoagulant (B01AF)": "B01AF", "Diurétique de l'anse (C03C)": "C03C", "Antidiabétique (A10)": "A10"}
for nom, prefixe in MARQUEURS.items():
    poly[nom] = poly.index.isin(med_ok.loc[med_ok["code_atc"].str.startswith(prefixe), "id_assure"])
poly["age"] = assures_ok.set_index("id_assure")["age"].reindex(poly.index)
poly.groupby("traitement").agg(personnes=("hospitalise", "size"), age_moyen=("age", "mean"),
                               **{f"% {nom}": (nom, lambda s: 100 * s.mean()) for nom in MARQUEURS})

**Question (piège D, suite).** Que montre ce tableau ? Que signifie l'expression « confusion par l'indication » ? Quelles informations, absentes des données de remboursement, et quel type d'étude faudrait-il pour évaluer l'effet des associations d'antihypertenseurs ?

Les personnes traitées sont en moyenne plus âgées. On refait donc la comparaison par classe d'âge, pour séparer l'effet de l'âge de celui de l'hypertension.

In [ ]:
ax = sns.pointplot(data=recours, x="classe_age", y="medecin_generaliste", hue="groupe",
                   palette=COULEURS, hue_order=GROUPES, errorbar=None)
ax.set(title="Consultations chez le généraliste, à âge égal", xlabel="Classe d'âge", ylabel="Contacts moyens par personne")
ax.legend(title="");

In [ ]:
specialites = (contacts_maladie[contacts_maladie["type_prestation"] == "medecin_specialiste"]
               .merge(assures_ok[["id_assure", "groupe"]], on="id_assure"))
repartition = pd.crosstab(specialites["specialite"], specialites["groupe"], normalize="columns")[GROUPES]
ax = repartition.sort_values("HTA traitée").rename(index=libelle).plot.barh(color=COULEURS, width=0.75, figsize=(8, 5))
axe_pct(ax, "x")
ax.legend(title="")
ax.set(title="Répartition des consultations de spécialistes", xlabel="Part des consultations", ylabel="");

**Exercice 9. Des personnes traitées sans consultation ?** *(≈ 8 min)*

Un traitement antihypertenseur nécessite une ordonnance médicale. Calculer, selon la franchise, la part (en %) des personnes traitées pour hypertension n'ayant aucun contact de généraliste enregistré dans les données.

In [ ]:
## Exercice 9 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# hta_recours = recours[recours["groupe"] == "HTA traitée"].join(assures_ok.set_index("id_assure")["franchise_chf"])
# sans_generaliste = 100 * hta_recours.groupby(___)["medecin_generaliste"].apply(lambda s: (s == ___).mean())
# sans_generaliste

<details><summary><b>Indice</b></summary>

Grouper par la colonne de la franchise. Dans chaque groupe, on cherche la part des personnes dont le nombre de contacts de généraliste est nul.

</details>

**Questions (exercice 9)**
1. Quelles raisons médicales peuvent expliquer qu'une personne traitée n'ait aucune consultation de généraliste enregistrée ?
2. Pourquoi cette part augmente-t-elle avec la franchise ? Qu'en déduire pour la mesure du recours aux soins ?

**Questions**
1. Les personnes traitées recourent-elles davantage aux soins, et pour quels types de prestations ?
2. La différence persiste-t-elle à âge égal ? Quels autres facteurs pourraient l'expliquer ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

**Analyse complémentaire : continuité des soins.** La variable `id_prestataire` permet de savoir si une personne consulte toujours le même généraliste.

In [ ]:
generalistes = prest_ok[(prest_ok["type_prestation"] == "medecin_generaliste") & (prest_ok["cause"] == "maladie")
                        & (prest_ok["cout_chf"] > 0)]
continuite = generalistes.groupby("id_assure").agg(jours=("date_prestation", "nunique"),
                                                   medecins=("id_prestataire", "nunique"))
continuite = continuite[continuite["jours"] >= 3].join(assures_ok.set_index("id_assure")["groupe"])
print("Personnes avec au moins 3 consultations de généraliste n'ayant vu qu'un seul généraliste :")
continuite.groupby("groupe")["medecins"].apply(lambda s: (s == 1).mean()).loc[GROUPES].map("{:.0%}".format).to_frame("")

**Question.** Pourquoi la continuité des soins est-elle un enjeu particulier pour les personnes atteintes d'une maladie chronique ? Quel lien avec les modèles d'assurance « médecin de famille » ?

### 4.5 Coûts

Le coût comprend les prestations de cause « maladie » et les médicaments, décomptés en 2024 à la charge de l'assurance obligatoire des soins.

In [ ]:
prest_maladie = prest_ok[prest_ok["cause"] == "maladie"]
couts = prest_maladie.pivot_table(index="id_assure", columns="type_prestation", values="cout_chf", aggfunc="sum")
couts["med_antihypertenseurs"] = antihta.groupby("id_assure")["cout_chf"].sum()
couts["med_autres"] = med_ok[~med_ok.index.isin(antihta.index)].groupby("id_assure")["cout_chf"].sum()
couts = couts.reindex(assures_ok["id_assure"]).fillna(0)

POSTES = list(couts.columns)
couts["total"] = couts[POSTES].sum(axis=1)
couts = couts.join(assures_ok.set_index("id_assure")[["groupe", "classe_age", "franchise_chf", "motif_fin_couverture"]])

(couts.groupby("groupe")["total"]
 .describe(percentiles=[0.5, 0.9])[["count", "mean", "50%", "90%", "max"]]
 .rename(columns={"count": "personnes", "mean": "moyenne", "50%": "médiane", "90%": "90e centile"})
 .loc[GROUPES].round(0))

In [ ]:
ax = sns.histplot(data=couts[couts["total"] > 0], x="total", hue="groupe", hue_order=GROUPES,
                  palette=COULEURS, log_scale=True, bins=40, stat="density", common_norm=False, element="step")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: chf(v)))
ax.get_legend().set_title("")
ax.set(title="Distribution du coût annuel par personne (échelle logarithmique)", xlabel="CHF", ylabel="");

In [ ]:
NOMS = {"stationnaire": "Hospitalisation", "ambulatoire_hospitalier": "Ambulatoire hospitalier",
        "medecin_specialiste": "Spécialiste", "medecin_generaliste": "Généraliste", "laboratoire": "Laboratoire",
        "med_autres": "Médicaments : autres", "med_antihypertenseurs": "Médicaments : antihypertenseurs"}
ORDRE = ["Hospitalisation", "Ambulatoire hospitalier", "Spécialiste", "Généraliste", "Laboratoire",
         "Autres prestations", "Médicaments : autres", "Médicaments : antihypertenseurs"]   # ordre et couleurs fixes
postes = couts.groupby("groupe")[POSTES].mean().loc[GROUPES].T.groupby(lambda c: NOMS.get(c, "Autres prestations")).sum().T
postes = postes.reindex(columns=ORDRE, fill_value=0)
ax = postes.plot.bar(stacked=True, color=CATEGORIELLES, width=0.6, figsize=(7, 5), edgecolor="white", linewidth=1)
axe_chf(ax)
ax.tick_params(axis="x", rotation=0)
poignees, etiquettes = ax.get_legend_handles_labels()
ax.legend(poignees[::-1], etiquettes[::-1], bbox_to_anchor=(1, 1), loc="upper left")   # même ordre que l'empilement
ax.set(title="Coût moyen par personne et par poste", xlabel="", ylabel="CHF");

part_antihta = postes.loc["HTA traitée", "Médicaments : antihypertenseurs"] / postes.loc["HTA traitée"].sum()
print(f"Antihypertenseurs = {part_antihta:.1%} du coût moyen des personnes traitées")

In [ ]:
ax = sns.pointplot(data=couts, x="classe_age", y="total", hue="groupe", palette=COULEURS,
                   hue_order=GROUPES, estimator="mean", errorbar=None)
axe_chf(ax)
ax.set(title="Coût moyen par personne, à âge égal", xlabel="Classe d'âge", ylabel="CHF")
ax.legend(title="");

In [ ]:
poids = couts.groupby("groupe").agg(personnes=("total", "size"), couts=("total", "sum"))
poids = (poids / poids.sum()).loc[GROUPES]
ax = poids.T.rename(index={"personnes": "Part des assuré·es", "couts": "Part des coûts"}).plot.barh(
    stacked=True, color=COULEURS, width=0.6, figsize=(8, 2.5), edgecolor="white", linewidth=1)
ax.invert_yaxis()
ax.bar_label(ax.containers[0], labels=[f"{v:.0%}" for v in poids.loc["HTA traitée"]], label_type="center", color="white")
axe_pct(ax, "x")
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Part des personnes traitées dans l'effectif et dans les coûts", xlabel="", ylabel="");

plus_couteux = couts.sort_values("total", ascending=False).iloc[:len(couts) // 10]
part_hta_top10 = (plus_couteux["groupe"] == "HTA traitée").mean()
print(f"Personnes traitées pour HTA parmi les 10 % les plus coûteux : {part_hta_top10:.0%} "
      f"(contre {hta.mean():.0%} de l'ensemble des assuré·es)")

**Exercice 10. Coût à âge égal : standardisation directe** *(≈ 10 min)*

Les personnes traitées pour hypertension sont plus âgées, et le coût augmente avec l'âge. La standardisation directe estime le coût moyen qu'aurait chaque groupe s'il avait la structure d'âge d'une population de référence, ici l'ensemble des assuré·es : les coûts moyens de chaque classe d'âge sont pondérés par la part de cette classe dans la population de référence. Compléter le calcul, puis comparer le rapport des coûts brut et standardisé.

In [ ]:
## Exercice 10 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# moyennes_age = couts.groupby(["classe_age", "groupe"], observed=True)["total"].mean().unstack()[GROUPES]
# structure = assures_ok["classe_age"].value_counts(normalize=True).reindex(moyennes_age.index)
# standardise = moyennes_age.mul(___, axis=0).sum()
# brut = couts.groupby("groupe")["total"].mean()[GROUPES]
# print(f"Coût moyen brut          : traitées CHF {chf(brut.iloc[0])}, autres CHF {chf(brut.iloc[1])}, rapport {brut.iloc[0] / brut.iloc[1]:.1f}")
# print(f"Coût moyen standardisé   : traitées CHF {chf(standardise.iloc[0])}, autres CHF {chf(standardise.iloc[1])}, rapport {standardise.iloc[0] / standardise.iloc[1]:.1f}")

<details><summary><b>Indice</b></summary>

Les poids sont les parts des classes d'âge dans l'ensemble des assuré·es, calculées à la ligne précédente.

</details>

**Questions (exercice 10)**
1. Le rapport des coûts diminue-t-il une fois l'âge pris en compte ?
2. L'écart restant mesure-t-il le coût de l'hypertension ? Quels autres facteurs l'expliquent ?

**Questions**
1. Compte tenu de la distribution des coûts, quel indicateur (moyenne ou médiane) convient-il de présenter à un·e décideur·e, et dans quel but ?
2. Quelle part des coûts des personnes traitées est attribuable aux antihypertenseurs ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### Analyses complémentaires

In [ ]:
# Personnes non assurées toute l'année
couts["motif"] = couts["motif_fin_couverture"].fillna("assuré·e toute l'année")
couts.groupby("motif")["total"].agg(["size", "mean", "median"]).round(0)

**Question.** Que suggèrent ces résultats concernant les coûts en fin de vie ? Quel biais introduit la comparaison de personnes assurées pendant des durées différentes ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 5. Synthèse et rédaction de la note

Le tableau suivant rassemble les principaux résultats. Les valeurs dépendent des choix effectués dans les sections précédentes.

In [ ]:
# Synthèse des principaux résultats
g = couts.groupby("groupe")["total"]
chiffres = pd.Series({
    "Personnes traitées pour hypertension": f"{hta.sum():,} ({hta.mean():.1%} des assuré·es)",
    "Âge moyen (traitées / autres)": f"{profil.loc['age_moyen', 'HTA traitée']:.0f} / {profil.loc['age_moyen', 'Autres assuré·es']:.0f} ans",
    "Part de femmes parmi les traitées": f"{profil.loc['pct_femmes', 'HTA traitée']:.0f}%",
    "Traitées avec ≥ 2 classes": f"{(par_personne['nb_classes'] >= 2).mean():.0%}",
    "Généraliste, contacts/an (traitées / autres)":
        f"{moyennes.loc['medecin_generaliste', 'HTA traitée']:.1f} / {moyennes.loc['medecin_generaliste', 'Autres assuré·es']:.1f} par an",
    "Hospitalisées ≥ 1 fois (traitées / autres)":
        f"{recours.groupby('groupe')['hospitalise'].mean()['HTA traitée']:.0%} / {recours.groupby('groupe')['hospitalise'].mean()['Autres assuré·es']:.0%}",
    "Coût moyen (traitées / autres)": f"{g.mean()['HTA traitée']:,.0f} / {g.mean()['Autres assuré·es']:,.0f} CHF",
    "Coût médian (traitées / autres)": f"{g.median()['HTA traitée']:,.0f} / {g.median()['Autres assuré·es']:,.0f} CHF",
    "Part des coûts totaux portée par les traitées": f"{poids.loc['HTA traitée', 'couts']:.0%}",
    "Part des traitées parmi les 10 % les plus coûteux": f"{part_hta_top10:.0%}",
    "Part des antihypertenseurs dans leur coût": f"{part_antihta:.1%}",
}).str.replace(",", "'")
chiffres.to_frame("Valeur")

### Structure de la note

**Destinataire :** responsable de la santé publique cantonale.
**Objet :** personnes traitées pour hypertension artérielle : profil, recours aux soins et coûts.

1. **Population concernée.** Effectif, proportion et principales caractéristiques.
2. **Traitements et recours aux soins.** Éléments distinguant les personnes traitées des autres assuré·es.
3. **Coûts associés.** Ordres de grandeur, principaux déterminants et place dans la concentration des dépenses.
4. **Messages principaux.** Deux ou trois, formulés en vue de la décision.
5. **Limite d'interprétation.** La principale limite et ses implications pour la décision (les pièges d'interprétation A à D en donnent des exemples).

Une page, avec au plus un graphique. Des chiffres arrondis, et pas de jargon d'analyse de données.